# Advanced Analysis: IV Gaussian Filter Data

This notebook performs comprehensive analysis on the Gaussian-filtered IV H-scan data.

## Analysis Components:
1. I(H) curves at fixed voltages
2. 2D color map (H vs V with I as color)
3. Voltage offset vs magnetic field
4. Quality metrics (residual RMS)
5. Hysteresis analysis
6. Export results

## 1. Setup & Data Loading

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
from scipy.interpolate import griddata
from scripts.IV_Hscan_gaussian import load_dataframe, get_current_at_voltage, get_asymmetric_current_at_voltage

In [ ]:
# Load DataFrame
df_path = PROJECT_ROOT / r"output/IV_H_scans/dataframes/b_scans/IV_gaussian_10K.pkl"
df = load_dataframe(df_path)

print(f"\nDataFrame loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nH field range: {df['H'].min():.4f} to {df['H'].max():.4f} T")
print(f"Number of measurements: {len(df)}")
temperature = int(df['temperature'].iloc[0] if 'temperature' in df.columns else 'N/A')
print(f"Temperature: {temperature} K")

In [ ]:
output_path = df_path.parent 
print(f"\nOutput path for figures: {output_path}")

In [ ]:
# Display first few rows (scalar columns only)
scalar_cols = ['timestamp', 'H', 'Hx', 'Hy', 'Hz', 'sigma', 'v_offset', 'residual_rms']
if 'temperature' in df.columns:
    scalar_cols.append('temperature')

## 2. I(H) Curves at Fixed Voltages

Plot current vs magnetic field at specific bias voltages to observe magnetoresistance behavior.

In [ ]:

import matplotlib.cm as cm

# Assume df and get_current_at_voltage are defined from previous cells

voltages_to_plot = np.linspace(-1, 1, 51)

print(f"Will plot I(H) curves at {len(voltages_to_plot)} voltages.")

# --- Step 1: Set up the colormap and normalization ---
# The RdBu_r colormap is great here: Red for positive V, Blue for negative V.
cmap = cm.get_cmap('RdBu_r') 
# Create a normalizer to map the voltage range [-1, 1] to the [0, 1] interval for the colormap.
norm = plt.Normalize(vmin=voltages_to_plot.min(), vmax=voltages_to_plot.max())

# --- Step 2: Plot the data ---
fig, ax = plt.subplots(figsize=(12, 7))

for V_val in voltages_to_plot:
    # Get current at this voltage using filtered data
    I_vs_H = get_current_at_voltage(df, V_val, use_filtered=True)
    
    # Calculate the color for this specific voltage
    color = cmap(norm(V_val))
    
    # Plot the line, setting the color explicitly and removing the 'label'
    ax.plot(I_vs_H['H'], I_vs_H['I_at_V'] * 1e6, 'o-', 
            color=color, markersize=4, linewidth=2, alpha=0.8)

# --- Step 3: Add the color bar ---
# Create a ScalarMappable object that the colorbar can use
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# Add the colorbar to the figure, linking it to the axes
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Voltage (V)', fontsize=14)

# --- Step 4: Final plot formatting ---
ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Current (µA)', fontsize=14)
ax.set_title(f'I(H) Curves at Fixed Voltages (Gaussian Filtered)\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
# ax.legend(...) # The legend is no longer needed

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.cm as cm

# Assume df and get_current_at_voltage are defined from previous cells

voltages_to_plot = np.linspace(-1, 1, 51)

print(f"Will plot I(H) curves at {len(voltages_to_plot)} voltages.")

# --- Step 1: Set up the colormap and normalization ---
# The RdBu_r colormap is great here: Red for positive V, Blue for negative V.
cmap = cm.get_cmap('RdBu_r') 
# Create a normalizer to map the voltage range [-1, 1] to the [0, 1] interval for the colormap.
norm = plt.Normalize(vmin=voltages_to_plot.min(), vmax=voltages_to_plot.max())

# --- Step 2: Plot the data ---
fig, ax = plt.subplots(figsize=(12, 7))

for V_val in voltages_to_plot:
    # Get current at this voltage using filtered data
    I_vs_H = get_asymmetric_current_at_voltage(df, V_val)
    
    # Calculate the color for this specific voltage
    color = cmap(norm(V_val))
    
    # Plot the line, setting the color explicitly and removing the 'label'
    ax.plot(I_vs_H['H'], I_vs_H['I_asym_at_V'] * 1e6, 'o-', 
            color=color, markersize=4, linewidth=2, alpha=0.8)

# --- Step 3: Add the color bar ---
# Create a ScalarMappable object that the colorbar can use
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# Add the colorbar to the figure, linking it to the axes
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Voltage (V)', fontsize=14)

# --- Step 4: Final plot formatting ---
ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Current (µA)', fontsize=14)
ax.set_title(f'I(H) Curves at Fixed Voltages (Gaussian Filtered)\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
# ax.legend(...) # The legend is no longer needed

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
voltage_points = np.linspace(-1.0, 1.0, 200) # Reduced points for faster testing if needed

# Field thresholds
H_low_threshold = 0.1   # T - low field region
H_high_threshold = 0.5  # T - high field region

# Lists to store results
tmr_values = []
tmr_errors = []

for V_target in voltage_points:
    # Get I at all H values for this voltage
    data = get_current_at_voltage(df, V_target, use_filtered=True)
    
    H_vals = data['H'].values
    I_vals = np.abs(data['I_at_V'].values)
    
    # --- I_min Calculation with Error ---
    low_field_mask = np.abs(H_vals) < H_low_threshold
    if low_field_mask.sum() > 0:
        currents_at_low_field = I_vals[low_field_mask]
        I_min = np.mean(currents_at_low_field)
        # The error is the standard deviation of the points used for the mean
        I_min_err = np.std(currents_at_low_field)
    else:
        I_min, I_min_err = np.nan, np.nan
    
    # --- I_max Calculation with Error ---
    high_field_mask = np.abs(H_vals) > H_high_threshold
    if high_field_mask.sum() > 0:
        currents_at_high_field = I_vals[high_field_mask]
        I_max = np.mean(currents_at_high_field)
        # The error is the standard deviation of the points used for the mean
        I_max_err = np.std(currents_at_high_field)
    else:
        I_max, I_max_err = np.nan, np.nan
    
    # --- TMR Ratio and Error Propagation ---
    if I_min > 0 and I_max > 0 and not np.isnan(I_min) and not np.isnan(I_max):
        tmr = I_max / I_min
        # Standard error propagation for division: TMR = I_max / I_min
        tmr_error = tmr * np.sqrt((I_max_err / I_max)**2 + (I_min_err / I_min)**2)
        
        tmr_values.append(tmr)
        tmr_errors.append(tmr_error)
    else:
        tmr_values.append(np.nan)
        tmr_errors.append(np.nan)

# Convert lists to numpy arrays for easier plotting
tmr_values = np.array(tmr_values)
tmr_errors = np.array(tmr_errors)

print(f"\nTMR calculation complete!")
print(f"Voltage range: {voltage_points.min():.2f} to {voltage_points.max():.2f} V")
print(f"TMR ratio range: {np.nanmin(tmr_values):.3f} to {np.nanmax(tmr_values):.3f}")


In [ ]:
plt.figure(figsize=(10, 6))

# Plot the TMR ratio as a solid line
plt.plot(voltage_points, tmr_values, '-', color='dodgerblue', linewidth=2, label='TMR Ratio')

# Add a shaded region to represent the propagated error
plt.fill_between(voltage_points, 
                 tmr_values - tmr_errors, 
                 tmr_values + tmr_errors,
                 color='dodgerblue', alpha=0.2, label='Propagated Error')

# --- Formatting the Plot ---
plt.xlabel("Voltage (V)", fontsize=14)
plt.ylabel("TMR Ratio (I_max / I_min)", fontsize=14)
plt.title("TMR Ratio vs. Applied Voltage with Propagated Error", fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)

# Set a sensible y-axis limit
plt.ylim(bottom=0) 

plt.tight_layout()
plt.show()




In [ ]:
voltage_points = np.linspace(-1.0, 1.0, 200) # Reduced points for faster testing if needed

# Field thresholds
H_low_threshold = 0.1   # T - low field region
H_high_threshold = 0.5  # T - high field region

# Lists to store results
tmr_values = []
tmr_errors = []

for V_target in voltage_points:
    # Get I at all H values for this voltage
    data = get_asymmetric_current_at_voltage(df, V_target)
    
    H_vals = data['H'].values
    I_vals = np.abs(data['I_asym_at_V'].values)
    
    # --- I_min Calculation with Error ---
    low_field_mask = np.abs(H_vals) < H_low_threshold
    if low_field_mask.sum() > 0:
        currents_at_low_field = I_vals[low_field_mask]
        I_min = np.mean(currents_at_low_field)
        # The error is the standard deviation of the points used for the mean
        I_min_err = np.std(currents_at_low_field)
    else:
        I_min, I_min_err = np.nan, np.nan
    
    # --- I_max Calculation with Error ---
    high_field_mask = np.abs(H_vals) > H_high_threshold
    if high_field_mask.sum() > 0:
        currents_at_high_field = I_vals[high_field_mask]
        I_max = np.mean(currents_at_high_field)
        # The error is the standard deviation of the points used for the mean
        I_max_err = np.std(currents_at_high_field)
    else:
        I_max, I_max_err = np.nan, np.nan
    
    # --- TMR Ratio and Error Propagation ---
    if I_min > 0 and I_max > 0 and not np.isnan(I_min) and not np.isnan(I_max):
        tmr = I_max / I_min
        # Standard error propagation for division: TMR = I_max / I_min
        tmr_error = tmr * np.sqrt((I_max_err / I_max)**2 + (I_min_err / I_min)**2)
        
        tmr_values.append(tmr)
        tmr_errors.append(tmr_error)
    else:
        tmr_values.append(np.nan)
        tmr_errors.append(np.nan)

# Convert lists to numpy arrays for easier plotting
asym_tmr_values = np.array(tmr_values)
asym_tmr_errors = np.array(tmr_errors)

print(f"\nTMR calculation complete!")
print(f"Voltage range: {voltage_points.min():.2f} to {voltage_points.max():.2f} V")
print(f"TMR ratio range: {np.nanmin(tmr_values):.3f} to {np.nanmax(tmr_values):.3f}")

In [ ]:


plt.figure(figsize=(12, 7))

# Plot the TMR ratio as a solid line
plt.plot(voltage_points, asym_tmr_values, '-', color='dodgerblue', linewidth=2, label='TMR Ratio')

# Add a shaded region to represent the propagated error
plt.fill_between(voltage_points, 
                 asym_tmr_values - asym_tmr_errors, 
                 asym_tmr_values + asym_tmr_errors,
                 color='dodgerblue', alpha=0.2, label='Propagated Error')

# --- Formatting the Plot ---
plt.xlabel("Voltage (V)", fontsize=14)
plt.ylabel("TMR Ratio (I_max / I_min)", fontsize=14)
plt.title("TMR Ratio vs. Applied Voltage with Propagated Error", fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)

# Set a sensible y-axis limit
plt.ylim(bottom=0) 

plt.tight_layout()
plt.show()


In [ ]:
data_to_save = {
    'Voltage (V)': voltage_points,
    'TMR_Ratio': tmr_values,
    'TMR_Error': tmr_errors,
    'asym_TMR_Ratio': asym_tmr_values,
    'asym_TMR_Error': asym_tmr_errors
}

df_results = pd.DataFrame(data_to_save)


try:
    # This is the line you should use
    filename = f"TMR_ratio_vs_V_{temperature}K.csv"
except NameError:
    # Placeholder in case 'metadata' is not defined in this context
    print("Warning: 'metadata' object not found. Using a placeholder temperature.")
    temperature_placeholder = "295" 
    filename = f"TMR_ratio_vs_V_{temperature_placeholder}K.csv"



df_results.to_csv(output_path / filename, index=False)

print(f"Data successfully saved to: {filename}")

# You can display the head of the DataFrame to double-check it
print("\nFirst 5 rows of the data saved:")
print(df_results.head())

## 6. IV Characteristics at Different Magnetic Fields

Plot I(V) curves at selected H field values to show how current evolves with field.

In [ ]:
# Select representative H field values
# Choose fields spanning the range: min, low, zero, high, max
H_values_to_plot = [
    df['H'].min(),
    -0.4,
    0.0,
    0.4,
    df['H'].max()
]

print(f"Will plot I(V) at {len(H_values_to_plot)} magnetic fields:")
for H in H_values_to_plot:
    print(f"  H = {H:+.4f} T")

# Plot IV characteristics
fig, ax = plt.subplots(figsize=(12, 7))

# Color map for different H values
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(H_values_to_plot)))

for H_val, color in zip(H_values_to_plot, colors):
    # Find row closest to this H value
    idx = (df['H'] - H_val).abs().idxmin()
    row = df.loc[idx]
    
    V = row['voltage_smooth']
    I = row['current_smooth']
    
    ax.plot(V, I * 1e6, '-', 
            color=color, linewidth=2, 
            label=f'H = {row["H"]:+.4f} T', alpha=0.8)

ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Voltage (V)', fontsize=14)
ax.set_ylabel('Current (µA)', fontsize=14)
ax.set_title(f'IV Characteristics at Different Magnetic Fields (Gaussian Filtered)\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate TMR ratio across voltage range
voltage_points = np.linspace(-1.0, 1.0, 500)

# Field thresholds
H_low_threshold = 0.1   # T - low field region
H_high_threshold = 0.5  # T - high field region

TMR_ratio = []

print(f"Calculating TMR ratios for {len(voltage_points)} voltage points...")
print(f"Using field thresholds:")
print(f"  I_min region: |H| < {H_low_threshold} T")
print(f"  I_max region: |H| > {H_high_threshold} T")

for V_target in voltage_points:
    # Get I at all H values for this voltage
    data = get_current_at_voltage(df, V_target, use_filtered=True)
    
    H_vals = data['H'].values
    I_vals = np.abs(data['I_at_V'].values)  # Take absolute value
    
    # Calculate I_min: mean of |I| in low field region
    low_field_mask = np.abs(H_vals) < H_low_threshold
    if low_field_mask.sum() > 0:
        I_min = np.mean(I_vals[low_field_mask])
    else:
        I_min = np.nan
    
    # Calculate I_max: mean of |I| in high field region
    high_field_mask = np.abs(H_vals) > H_high_threshold
    if high_field_mask.sum() > 0:
        I_max = np.mean(I_vals[high_field_mask])
    else:
        I_max = np.nan
    
    # TMR ratio
    if I_min > 0 and not np.isnan(I_min) and not np.isnan(I_max):
        TMR_ratio.append(I_max / I_min)
    else:
        TMR_ratio.append(np.nan)

TMR_ratio = np.array(TMR_ratio)

print(f"\nTMR calculation complete!")
print(f"Voltage range: {voltage_points.min():.2f} to {voltage_points.max():.2f} V")
print(f"TMR ratio range: {np.nanmin(TMR_ratio):.3f} to {np.nanmax(TMR_ratio):.3f}")

In [ ]:
# Select representative H field values
# Choose fields spanning the range: min, low, zero, high, max
H_values_to_plot = [
    df['H'].min(),
    -0.4,
    0.0,
    0.4,
    df['H'].max()
]

print(f"Will plot I(V) at {len(H_values_to_plot)} magnetic fields:")
for H in H_values_to_plot:
    print(f"  H = {H:+.4f} T")

# Plot IV characteristics
fig, ax = plt.subplots(figsize=(12, 7))

# Color map for different H values
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(H_values_to_plot)))

for H_val, color in zip(H_values_to_plot, colors):
    # Find row closest to this H value
    idx = (df['H'] - H_val).abs().idxmin()
    row = df.loc[idx]
    
    V = row['voltage_smooth']
    I = row['current_smooth']
    
    ax.plot(V, I * 1e6, '-', 
            color=color, linewidth=2, 
            label=f'H = {row["H"]:+.4f} T', alpha=0.8)

ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Voltage (V)', fontsize=14)
ax.set_ylabel('Current (µA)', fontsize=14)
ax.set_title(f'IV Characteristics at Different Magnetic Fields (Gaussian Filtered)\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot residual RMS vs H field
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['H'], df['residual_rms'] * 1e6, 'o-', markersize=5, linewidth=2, color='purple')

ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Residual RMS (µA)', fontsize=14)
ax.set_title(f'Filtering Quality vs Magnetic Field\n' +
             f'Gaussian Filter σ = {df["sigma"].iloc[0]:.1f}',
             fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\n=== Residual RMS Statistics ===")
print(f"Mean:   {df['residual_rms'].mean() * 1e6:.4f} µA")
print(f"Std:    {df['residual_rms'].std() * 1e6:.4f} µA")
print(f"Min:    {df['residual_rms'].min() * 1e6:.4f} µA")
print(f"Max:    {df['residual_rms'].max() * 1e6:.4f} µA")

In [ ]:
# Plot V_offset vs H
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['H'], df['v_offset'], 'o-', markersize=5, linewidth=2, color='green')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Voltage Offset (V)', fontsize=14)
ax.set_title(f'Voltage Offset vs Magnetic Field\n' +
             f'Temperature: {df["temperature"].iloc[0] if "temperature" in df.columns else "N/A"} K',
             fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\n=== Voltage Offset Statistics ===")
print(f"Mean:   {df['v_offset'].mean():.6f} V")
print(f"Std:    {df['v_offset'].std():.6f} V")
print(f"Min:    {df['v_offset'].min():.6f} V")
print(f"Max:    {df['v_offset'].max():.6f} V")
print(f"Range:  {df['v_offset'].max() - df['v_offset'].min():.6f} V")